# 03 - Deploy Multi-Model Serving Endpoint

Deploy all 5 personalized user models to a **single** Model Serving endpoint.
Each model is deployed as a separate served entity that can be invoked by name.

**Endpoint:** `hyper-personalization-models`

**Served Entities:**
- `user-alice-model` → Electronics repeat purchase predictor
- `user-bob-model` → Fitness repeat purchase predictor
- `user-carol-model` → Kitchen repeat purchase predictor
- `user-dave-model` → Books repeat purchase predictor
- `user-eve-model` → Fashion repeat purchase predictor

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedModelInput,
    TrafficConfig,
    Route,
)
import time

w = WorkspaceClient()

# Parameters
dbutils.widgets.text("catalog", "custom_ml", "Catalog")  # Only hardcoded default
dbutils.widgets.text("schema", "hyper_personalization", "Schema")
dbutils.widgets.text("endpoint_name", "hyper-personalization-models", "Endpoint Name")
dbutils.widgets.text("user_ids", "user_alice,user_bob,user_carol,user_dave,user_eve", "User IDs (comma-separated)")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
ENDPOINT_NAME = dbutils.widgets.get("endpoint_name")
USER_IDS = [u.strip() for u in dbutils.widgets.get("user_ids").split(",")]

print(f"Catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")
print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Users: {USER_IDS}")

In [0]:
# Dynamically build model list from user_ids parameter
models = [
    {
        "name": f"{CATALOG}.{SCHEMA}.{uid}_model",
        "entity_name": f"{uid.replace('_', '-')}-model",
    }
    for uid in USER_IDS
]

# Verify all models exist and get latest versions
for m in models:
    versions = list(w.model_versions.list(full_name=m["name"]))
    if not versions:
        raise ValueError(f"No versions found for model: {m['name']}")
    latest_version = max(versions, key=lambda v: int(v.version))
    m["version"] = latest_version.version
    print(f"  {m['entity_name']}: {m['name']} v{m['version']}")

print(f"\nAll {len(models)} models verified!")

In [0]:
# Build served models config with traffic routing
# Reference: https://github.com/RamVegiraju/databricks-samples/blob/master/traditional-ml/ModelServing/Multi-Model-Serving/multi-model-serving-intro.ipynb
served_models = [
    ServedModelInput(
        model_name=m["name"],
        model_version=m["version"],
        name=m["entity_name"],
        workload_size="Small",
        scale_to_zero_enabled=True,
    )
    for m in models
]

# Traffic config: equal split across all models (required for multi-model endpoints)
# Individual models are still invoked by name regardless of traffic split
traffic_pct = 100 // len(models)
remainder = 100 - (traffic_pct * len(models))

routes = []
for i, m in enumerate(models):
    pct = traffic_pct + (1 if i < remainder else 0)  # Distribute remainder
    routes.append(Route(served_model_name=m["entity_name"], traffic_percentage=pct))

traffic_config = TrafficConfig(routes=routes)

print(f"Configured {len(served_models)} served models:")
for sm, route in zip(served_models, routes):
    print(f"  - {sm.name} (v{sm.model_version}) | traffic: {route.traffic_percentage}%")

In [0]:
# Create or update the endpoint
try:
    # Check if endpoint already exists
    existing = w.serving_endpoints.get(name=ENDPOINT_NAME)
    print(f"Endpoint '{ENDPOINT_NAME}' exists. Updating configuration...")
    
    w.serving_endpoints.update_config_and_wait(
        name=ENDPOINT_NAME,
        served_models=served_models,
        traffic_config=traffic_config,
    )
    print("Endpoint updated successfully!")
    
except Exception as e:
    if "RESOURCE_DOES_NOT_EXIST" in str(e) or "does not exist" in str(e).lower():
        print(f"Creating new endpoint: {ENDPOINT_NAME}")
        
        w.serving_endpoints.create_and_wait(
            name=ENDPOINT_NAME,
            config=EndpointCoreConfigInput(
                served_models=served_models,
                traffic_config=traffic_config,
            ),
        )
        print("Endpoint created successfully!")
    else:
        raise e

In [0]:
# Verify endpoint is ready
endpoint = w.serving_endpoints.get(name=ENDPOINT_NAME)
print(f"Endpoint: {endpoint.name}")
print(f"State: {endpoint.state}")
print(f"\nServed entities:")
if endpoint.config and endpoint.config.served_entities:
    for entity in endpoint.config.served_entities:
        print(f"  - {entity.name}: {entity.entity_name} v{entity.entity_version} (state: {entity.state})")

print(f"\n✅ Endpoint '{ENDPOINT_NAME}' is ready for inference!")
print(f"   Invoke specific models using the 'name' field in your request.")